# L7: Build a Crew to Tailor Job Applications

In this lesson, you will built your first multi-agent system.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [19]:
# Switch kernel to: Python 3.11 (crewAI course)
# Packages are installed in crew_job/.venv311 — re-run import cell after switching.
print("OK: use kernel Python 3.11 (crewAI course)")

OK: use kernel Python 3.11 (crewAI course)


In [20]:
# !pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [21]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [22]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [23]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [24]:
from pathlib import Path
# Tools temporarily off — agents get prepared texts via kickoff inputs.
# from crewai_tools import FileReadTool, ScrapeWebsiteTool, MDXSearchTool, SerperDevTool

RESUME_MD = "input/filled_cv.md"  # from prepare notebook → Markdown export
JOB_POSTING_TXT = "input/job_posting.txt"  # paste vacancy description here (plain text)

# Prepared texts for {resume_text} / {job_posting_text} placeholders
resume_text = Path(RESUME_MD).read_text(encoding="utf-8")
job_posting_text = Path(JOB_POSTING_TXT).read_text(encoding="utf-8")
print(f"Loaded resume_text ({len(resume_text)} chars), job_posting_text ({len(job_posting_text)} chars)")

Loaded resume_text (4563 chars), job_posting_text (6566 chars)


- Uncomment and run the cell below if you wish to view `input/fake_resume.md` in the notebook.

In [25]:
from IPython.display import Markdown, display
display(Markdown("input/filled_cv.md"))

# Yuliia Antonova

**Obtain AI Data Engineer position**

Lengerich, Germany · yuliia.antonova1@gmail.com · +49 1759415902

- [Xing](https://www.xing.com/profile/Yuliia_Antonova)
- [GitHub](https://github.com/YuliiaAntonova)
- [LinkedIn](https://www.linkedin.com/in/yuliya25/)

## Skills

**Languages & Frameworks:** Python, PySpark, Pandas, Flask, Agentic AI

**Databases & Data:** SQL (DML), Oracle, Postgres, Snowflake

**Cloud & Data:** AWS Glue, Step Functions, Athena, Lambda, Aurora, OpenSearch, Databricks, Delta Lake

**Tools & BI:** Git (GitHub, GitLab), JIRA, PyCharm

**Domain:** Finance, Logistics, Manufacturing

## Experience

### Big Data Engineer — Trinetix Inc.
*Feb 2024 – Present*

Development of an AWS-based analytics platform for enterprise clients, supporting financial calculations, TB-scale data processing, and complex joins across ingestion, raw, derived, and data mart layers.

- Developed and optimized ETL pipelines using AWS Glue and PySpark for efficient data processing.
- Built data marts using Amazon Aurora and OpenSearch, improving performance with batch processing, chunked writes, and index optimization.
- Reduced AWS costs by moving RAW layer processing from AWS Glue to AWS Lambda, avoiding unnecessary Spark cluster runs.
- Built ETL workflows using AWS Step Functions to manage Glue, Lambda, and SNS, improving execution speed through parallel processing.
- Developed and optimized Amazon Redshift stored procedures for managing entity hierarchies and improving query performance.
- Developed an Agentic AI system to automate entity matching testing and reduce manual effort.
- Integrated AI agents with databases and APIs to retrieve data and validate matching results.
- Used AI agents to evaluate matching accuracy, identify errors, and analyze mismatched entities.

### Data Engineer — Independent Contractor
*Aug 2023 – Feb 2024 · Lengerich, Germany*

Modernization of a legacy data platform into a scalable AWS-based lakehouse architecture to support efficient data integration, transformation, and analytical workloads.

- Developed a multi-agent AI system using CrewAI and Python to automate project planning, task estimation, and resource allocation.
- Built AI workflows where multiple agents work together to break down tasks, estimate resources, and generate structured results.
- Built scalable ETL pipelines using AWS Glue to integrate data from multiple sources into the lakehouse.
- Automated data workflows using AWS Step Functions and Lambda, reducing manual work and improving processing efficiency.
- Optimized data models and schemas to improve query performance and simplify data access.
- Improved pipeline reliability by adding error handling, automatic retries, and monitoring.
- Improved ETL performance and data quality by fixing slow jobs and data processing errors.

### Data Engineer — HRForecast
*Feb 2022 – Jul 2023 · Lengerich, Germany*
Developed a scalable AWS-based data platform for recruitment analytics and predictions using Ray, Delta Lake, and Apache Airflow, processing millions of records daily.

- Built scalable data pipelines using Ray on AWS, improving the speed and efficiency of recruitment data processing.
- Optimized Delta Lake tables for faster data transformations and efficient data storage.
- Automated data workflows using Apache Airflow, improving pipeline reliability and reducing manual work.
- Improved data quality and pipeline stability by implementing automated tests with pytest.

## Earlier Experience

- **2021 – 2022** — Python Developer, Auchan Retail (Python)
- **2015 – 2021** — Business Data Analyst; Software Engineer, Energozbyt JSC Ukrainian railways

## Selected Projects

### [strategy-driven-etl](https://github.com/YuliiaAntonova/strategy-driven-etl)

- Designed and developed a locally runnable, configuration-driven ETL framework in Python, supporting multiple data sources (API, CSV, Snowflake), chunked processing, and configurable loading strategies (full refresh, incremental upsert, SCD Type 2) with PostgreSQL integration.

## Education

- **Master's degree in Economics and management** — Donetsk National Technical University
- **Python Advanced** — IT Education Academy (ITEA), Nov 2021 - Feb 2022

## Certifications

- DeepLearning.ai - Agentic AI (May 2026)
- DeepLearning.ai - Advanced Learning Algorithms (Aug 2026)
- DeepLearning.ai - Supervised Machine Learning: Regression and Classification (Dec 2025)

## Languages

- **English** — Upper-Intermediate (B2/C1)
- **German** — B1
- **Ukrainian / Russian** — Native / Fluent


## Creating Agents

In [26]:
# Agent 1: Researcher
researcher = Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on "
         "job posting to help job applicants",
    tools=[],  # job text via {job_posting_text}
    allow_delegation=False,
    max_iter=3,
    verbose=True,
    backstory=(
        "As a Job Researcher, your prowess in "
        "navigating and extracting critical "
        "information from job postings is unmatched."
        "Your skills help pinpoint the necessary "
        "qualifications and skills sought "
        "by employers, forming the foundation for "
        "effective application tailoring."
    )
)

In [27]:
# Agent 2: Profiler
profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do increditble research on job applicants "
         "to help them stand out in the job market",
    tools=[],  # resume via {resume_text} only
    allow_delegation=False,
    max_iter=3,
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect "
        "and synthesize information "
        "from diverse sources to craft comprehensive "
        "personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    )
)

In [28]:
# Agent 3: Resume Strategist
resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a "
         "resume stand out in the job market.",
    tools=[],  # use context from research + profile only
    allow_delegation=False,
    max_iter=3,
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you "
        "excel at refining resumes to highlight the most "
        "relevant skills and experiences, ensuring they "
        "resonate perfectly with the job's requirements."
    )
)

In [29]:
# Agent 4: Interview Preparer — temporarily disabled
# interview_preparer = Agent(
#     role="Engineering Interview Preparer",
#     goal="Create interview questions and talking points "
#          "based on the resume and job requirements",
#     tools=[],
#     allow_delegation=False,
#     max_iter=3,
#     verbose=True,
#     backstory=(
#         "Your role is crucial in anticipating the dynamics of "
#         "interviews. With your ability to formulate key questions "
#         "and talking points, you prepare candidates for success, "
#         "ensuring they can confidently address all aspects of the "
#         "job they are applying for."
#     )
# )

## Creating Tasks

In [30]:
# Task for Researcher Agent: Extract Job Requirements
research_task = Task(
    description=(
        "Analyze the prepared job posting text below. Extract key skills, "
        "experiences, and qualifications required. Do NOT search the web — "
        "use only this text.\n\n"
        "JOB POSTING TEXT:\n{job_posting_text}"
    ),
    expected_output=(
        "A structured list of job requirements, including necessary "
        "skills, qualifications, and experiences."
    ),
    agent=researcher,
    async_execution=True
)

In [31]:
# Task for Profiler Agent: Compile Comprehensive Profile
profile_task = Task(
    description=(
        "Compile a detailed personal and professional profile from the "
        "personal write-up and resume text below only. Do NOT browse GitHub "
        "or the web — projects and skills are already in the resume.\n\n"
        "PERSONAL WRITE-UP:\n{personal_writeup}\n\n"
        "RESUME TEXT:\n{resume_text}"
    ),
    expected_output=(
        "A comprehensive profile document that includes skills, "
        "project experiences, contributions, interests, and "
        "communication style."
    ),
    agent=profiler,
    async_execution=True
)

- You can pass a list of tasks as `context` to a task.
- The task then takes into account the output of those tasks in its execution.
- The task will not run until it has the output(s) from those tasks.

In [32]:
# Task for Resume Strategist Agent: Align Resume with Job Requirements
resume_strategy_task = Task(
    description=(
        "Using ONLY the profile and job requirements from previous tasks "
        "(provided as context), tailor the resume to highlight the most "
        "relevant areas. Do not invent facts. Do not call tools. Update "
        "every section — summary, work experience, skills, and education — "
        "so it matches the job posting.\n\n"
        "Opening/summary MUST be a short first-person Objective that names "
        "the target role and company from the job posting (e.g. 'To obtain "
        "a Senior Data Engineer position at Glimmer, …'), not a generic "
        "third-person 'Results-driven…' blurb. Keep company/role accurate "
        "from the job requirements context."
    ),
    expected_output=(
        "An updated resume that effectively highlights the candidate's "
        "qualifications and experiences relevant to the job."
    ),
    output_file="output/tailored_resume.md",
    context=[research_task, profile_task],
    agent=resume_strategist
)

In [33]:
# Task for Interview Preparer — temporarily disabled
# interview_preparation_task = Task(
#     description=(
#         "Create interview questions and talking points based only on the "
#         "tailored resume and job requirements from previous tasks (context). "
#         "Do not call tools. Help the candidate highlight how the resume "
#         "matches the job posting."
#     ),
#     expected_output=(
#         "A document containing key questions and talking points "
#         "that the candidate should prepare for the initial interview."
#     ),
#     output_file="output/interview_materials.md",
#     context=[research_task, profile_task, resume_strategy_task],
#     agent=interview_preparer
# )


## Creating the Crew

In [34]:
job_application_crew = Crew(
    agents=[researcher,
            profiler,
            resume_strategist],
            # interview_preparer],

    tasks=[research_task,
           profile_task,
           resume_strategy_task],
           # interview_preparation_task],

    verbose=True
)

2026-11-08 22:03:11,980 - 8536351296 - __init__.py-__init__:521 - WARNING: Overriding of current TracerProvider is not allowed


## Running the Crew

- Set the inputs for the execution of the crew.

In [35]:
from cv_inputs import build_crew_inputs_from_cv

_cv_bits = build_crew_inputs_from_cv("input/filled_cv.json")
job_application_inputs = {
    "personal_writeup": _cv_bits["personal_writeup"],
    "job_posting_text": job_posting_text,
    "resume_text": resume_text,
}
job_application_inputs

{'personal_writeup': 'Yuliia is an AI Data Engineer with 9+ years of experience, building scalable data platforms and modern data architectures. Key skills include Languages & Frameworks: Python, PySpark, Pandas, Flask; Databases & Data: SQL (DML), Oracle, Postgres, Snowflake; Cloud & Data: AWS Glue, Step Functions, Athena, Lambda; Tools & BI: Git (GitHub, GitLab), JIRA, PyCharm. Recently worked as Big Data Engineer at Trinetix Inc.. Focus: Development of an AWS-based analytics platform for enterprise clients, supporting financial calculations, TB-scale data processing, and complex joins across ingestion, raw, derived, and data mart layers.',
 'job_posting_text': "Senior Data Engineer (Full-time)\nAbout the job\nPregnancy, childbirth, and early parenthood bring about a lot of changes. And with each phase, new questions arise: small ones, big ones, practical ones, emotional ones. Glimmer helps to see more clearly what's happening right now, what's important, and what's right.\nWe combin

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [36]:
### this execution will take a few minutes to run
from run_usage import (
    reset_run_usage,
    record_crewai_usage,
    print_run_usage_summary,
    get_run_usage_summary,
)

reset_run_usage()
result = job_application_crew.kickoff(inputs=job_application_inputs)
record_crewai_usage(
    job_application_crew.usage_metrics,
    label="job_application_crew",
)
print_run_usage_summary()
get_run_usage_summary()


 [DEBUG]: == Working Agent: Tech Job Researcher
 [INFO]: == Starting Task: Analyze the prepared job posting text below. Extract key skills, experiences, and qualifications required. Do NOT search the web — use only this text.

JOB POSTING TEXT:
Senior Data Engineer (Full-time)
About the job
Pregnancy, childbirth, and early parenthood bring about a lot of changes. And with each phase, new questions arise: small ones, big ones, practical ones, emotional ones. Glimmer helps to see more clearly what's happening right now, what's important, and what's right.
We combine in-depth expert knowledge, a digital midwife consultation service, and practical tips in one app. This way, customers will find reliable answers, prenatal and postpartum recovery classes, and support all in one place. Accessible, compassionate, and tailored.
Glimmer is covered by over 80 health insurance providers and supports customers from their child's first heartbeat to their first steps. So they can navigate this journey

{'cursor': {'input_tokens': 14856,
  'output_tokens': 8142,
  'total_tokens': 22998,
  'calls': 9},
 'llama_parse': [],
 'llama_parse_credits_total': 0,
 'llm_calls': [{'call': 9,
   'label': 'job_application_crew',
   'backend': 'crewai',
   'input_tokens': 14856,
   'output_tokens': 8142,
   'total_tokens': 22998,
   'successful_requests': 9}]}

- Dislplay the generated `output/tailored_resume.md` file.

In [37]:
from IPython.display import Markdown, display
display(Markdown("output/tailored_resume.md"))

# Yuliia Antonova

**Contact Information:**  
- **Location:** Lengerich, Germany  
- **Email:** yuliia.antonova1@gmail.com  
- **Phone:** +49 1759415902  
- **Xing:** [Profile](https://www.xing.com/profile/Yuliia_Antonova)  
- **GitHub:** [Profile](https://github.com/YuliiaAntonova)  
- **LinkedIn:** [Profile](https://www.linkedin.com/in/yuliya25/)  

## Objective
To obtain a Senior Data Engineer position at Glimmer, where I can leverage my extensive experience in designing and maintaining scalable data pipelines and cloud solutions to drive business value and enhance data-driven decision-making.

## Skills
### Programming Languages
- **Python**: Proficient in developing scalable applications and data processing pipelines.
- **SQL**: Strong expertise in writing complex queries for data management and analytics.

### Database Technologies
- **PostgreSQL**: Extensive knowledge in data storage, retrieval, and performance optimization.
- **AWS RDS**: Experience in managing relational databases and ensuring data integrity and availability.

### Cloud Technologies
- **AWS (EC2, Glue, Lambda, Step Functions)**: Proficient in building and managing cloud-based data solutions, including ETL processes and workflow automation.
- **Data Pipeline Orchestration**: Skilled in utilizing AWS services to automate workflows and enhance operational efficiency.

### Data Engineering Practices
- **Data Modelling & Quality**: Solid understanding of data modeling principles and practices for maintaining data integrity and quality.
- **Pipeline Observability**: Familiarity with implementing monitoring and alerting processes to catch issues proactively.

### API Integrations
- **REST APIs**: Experience in developing and maintaining integrations with internal and external tools, ensuring seamless data exchange.

### Version Control & CI/CD
- **Git**: Proficient in version control practices, including collaborative development and code review processes.

### Additional Knowledge
- **Data Privacy & Security**: Awareness of GDPR compliance and best practices for handling sensitive data.
- **BI Tools**: Familiarity with tools such as AWS QuickSight for data visualization and reporting.

## Professional Experience
### Big Data Engineer — Trinetix Inc.  
*Feb 2024 – Present*  
- Led the development of an AWS-based analytics platform focused on financial data processing, achieving performance improvements for enterprise clients.
- Designed and optimized ETL pipelines using AWS Glue and Python, increasing data processing efficiency by 30%.
- Implemented data marts with PostgreSQL and enhanced retrieval performance through advanced indexing strategies.
- Developed automated workflows with AWS Step Functions, enabling faster execution of data processing tasks.

### Data Engineer — Independent Contractor  
*Aug 2023 – Feb 2024 · Lengerich, Germany*  
- Architected a scalable AWS-based lakehouse architecture, transforming legacy data platforms for improved analytics capabilities.
- Built robust ETL pipelines for integrating data from multiple sources, ensuring high data quality and accessibility.
- Automated data workflows using AWS Lambda, increasing reliability and reducing processing time by 40%.

### Data Engineer — HRForecast  
*Feb 2022 – Jul 2023 · Lengerich, Germany*  
- Developed a scalable data platform utilizing Apache Airflow for recruitment analytics, processing millions of records daily.
- Implemented data quality checks and monitoring processes, enhancing data integrity and reporting accuracy.
- Collaborated with cross-functional teams to understand data needs and deliver accurate datasets for decision-making.

## Education
- **Master's Degree in Economics and Management** — Donetsk National Technical University  
- **Advanced Python Certification** — IT Education Academy (ITEA), Nov 2021 - Feb 2022  

## Certifications
- **DeepLearning.ai - Agentic AI** (May 2026)  
- **DeepLearning.ai - Advanced Learning Algorithms** (Aug 2026)  
- **DeepLearning.ai - Supervised Machine Learning: Regression and Classification** (Dec 2025)  

## Languages
- **English** — Upper-Intermediate (B2/C1)  
- **German** — B1  
- **Ukrainian / Russian** — Native / Fluent  

## Interests
Yuliia is passionate about leveraging data engineering and cloud technologies to solve complex business challenges. She enjoys collaborating on innovative projects and continuously enhancing her technical skills to meet the evolving demands of the data landscape.

This updated resume effectively showcases Yuliia's relevant skills and experiences aligned with the Senior Data Engineer role at Glimmer, positioning her as a competitive candidate in the job market.

- Convert `output/tailored_resume.md` → `output/tailored_resume.json` (shape of `data/cv.json`), then render Harvard HTML.

In [38]:
import importlib
import json
import md_to_cv_json

importlib.reload(md_to_cv_json)
from md_to_cv_json import markdown_to_cv_json

tailored = markdown_to_cv_json(
    md_path="output/tailored_resume.md",
    template_path="data/cv.json",
    out_path="output/tailored_resume.json",
)
print("Wrote output/tailored_resume.json")
print(json.dumps(tailored.get("basics", {}), indent=2, ensure_ascii=False))

Wrote tailored_resume.json
{
  "name": "Yuliia Antonova",
  "headline": "Senior Data Engineer",
  "location": "Lengerich, Germany",
  "email": "yuliia.antonova1@gmail.com",
  "phone": "+49 1759415902",
  "summary": "To obtain a Senior Data Engineer position at Glimmer, where I can leverage my extensive experience in designing and maintaining scalable data pipelines and cloud solutions to drive business value and enhance data-driven decision-making.",
  "note": null,
  "links": [
    {
      "label": "Xing",
      "text": "Profile",
      "url": "https://www.xing.com/profile/Yuliia_Antonova"
    },
    {
      "label": "GitHub",
      "text": "Profile",
      "url": "https://github.com/YuliiaAntonova"
    },
    {
      "label": "LinkedIn",
      "text": "Profile",
      "url": "https://www.linkedin.com/in/yuliya25/"
    }
  ]
}


In [87]:
import importlib
from pathlib import Path

import render
from IPython.display import HTML, display

importlib.reload(render)
from render import render_cv_json_to_html

# Prefer helper.extract_html_content when available
try:
    from helper import extract_html_content
except ImportError:
    def extract_html_content(path):
        return Path(path).read_text(encoding="utf-8")

html_path = render_cv_json_to_html("output/tailored_resume.json", "output/tailored_resume.html")
print(f"HTML: {html_path.resolve()}")
display(HTML(extract_html_content(html_path)))

HTML: /Users/yuliiaantonova/PycharmProjects/PythonProject/crew_job/tailored_resume.html


- Dislplay the generated `output/interview_materials.md` file.

In [99]:
display(Markdown("output/interview_materials.md"))

### Interview Questions and Talking Points for Yuliia Antonova - Senior Data Engineer at Glimmer

#### Key Questions to Prepare For:

1. **SQL and Database Experience**
   - Can you describe your experience with SQL and how you have utilized it in your previous roles?
   - What databases have you worked with, and can you provide examples of how you've optimized queries in PostgreSQL?

2. **Cloud Technologies**
   - How have you used AWS in your past projects? Specifically, can you explain your experience with EC2 and RDS?
   - Can you detail a project where you built a scalable cloud data infrastructure on AWS? What challenges did you face, and how did you overcome them?

3. **ETL/ELT Pipeline Development**
   - Describe an ETL/ELT pipeline you designed and built. What tools did you use, and what was the outcome?
   - How do you ensure data quality and observability in your pipelines?

4. **Data Privacy and Security**
   - What experience do you have with handling sensitive or regulated data, particularly in relation to GDPR compliance?
   - Can you share an example of how you ensured data privacy while working on a data engineering project?

5. **API Integrations and Workflow Automation**
   - Can you explain your experience with API integrations? Give an example of a project where you implemented this.
   - How have you automated workflows in your previous roles? What tools did you use, and what improvements did automation bring?

6. **Collaboration and Cross-Functional Work**
   - Describe a situation where you worked with cross-functional teams (e.g., Marketing, Product, Finance). How did you gather their data needs and ensure they were met?
   - How do you communicate technical concepts to non-technical stakeholders?

7. **Version Control and Best Practices**
   - How do you use version control in your projects? Can you describe your experience with GitHub or GitLab?
   - What coding best practices do you follow to maintain code quality?

8. **Project Ownership and Autonomy**
   - Can you provide an example of a project where you had complete ownership? What was your approach to ensuring its success?

9. **Continuous Learning and Development**
   - How do you stay updated with emerging technologies and practices in data engineering?
   - What motivated you to pursue certifications in AI and machine learning, and how do you see this knowledge benefiting your role as a Data Engineer?

10. **Orchestration Tools and Automation**
    - Have you worked with orchestration tools like Apache Airflow or Dagster? If so, can you discuss a specific use case?
    - How have you utilized automation to improve data processes or workflows in your previous positions?

#### Talking Points:

- **Experience and Skills Alignment**: Highlight your 9+ years of experience in data engineering, focusing on designing, building, and maintaining scalable data architectures and ETL pipelines. Emphasize proficiency in SQL, Python, and AWS technologies, which align directly with Glimmer's requirements.

- **Hands-On Project Examples**: Use specific projects from your background, such as developing an AWS-based analytics platform or modernizing a legacy data platform into a lakehouse architecture, to demonstrate your technical skills and problem-solving abilities.

- **Data Privacy Expertise**: Discuss your understanding of data privacy and security, particularly your experience with GDPR compliance in handling sensitive health data. This is particularly relevant to Glimmer's focus on regulated data.

- **Collaboration and Communication**: Emphasize your ability to work closely with cross-functional teams to meet their data needs. Share examples of how you facilitated communication between technical and non-technical stakeholders.

- **Continuous Improvement Mindset**: Convey your passion for leveraging AI in data processing and your commitment to continuous learning. Mention your relevant certifications and how they enhance your contributions to data engineering projects.

- **Ownership and Autonomy**: Illustrate your high degree of autonomy and ownership in projects, discussing how you proactively address challenges and drive initiatives to successful outcomes.

By preparing these questions and talking points, Yuliia will be well-equipped to showcase her qualifications for the Senior Data Engineer position at Glimmer, ensuring a confident and compelling interview performance.